In [ ]:
import time
import os
import psutil
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, average_precision_score
from flaml import AutoML



In [12]:
df = pd.read_csv("../../data/creditcard.csv")

In [ ]:
# =========================
# COLUNAS
# =========================

X = df.drop(['Class'], axis=1)
y = df['Class'].astype(int)


In [ ]:
# =========================
# FUNÇÃO AUXILIAR DE RECURSOS
# =========================
def get_memory_usage_mb():
    """Retorna o uso atual de memória RAM do processo em Megabytes."""
    process = psutil.Process(os.getpid())
    return round(process.memory_info().rss / (1024 * 1024), 2)

# =========================
# INSTALAR FLAML
# =========================
# %pip install "flaml[automl]" psutil

for SEED in [42, 101, 223, 302, 404]:
    print(f"\n================================")
    print(f"EXECUTANDO COM SEED: {SEED}")
    print(f"================================")

    # =========================
    # DIVISÃO TREINO/TESTE
    # =========================
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
    )
    
    # =========================
    # TREINAMENTO COM FLAML + MÉTRICA COMPUTACIONAL
    # =========================
    automl = AutoML()
    settings = {
        "task": "classification",
        "metric": "f1", # Métrica orientada a custo/desbalanceamento
        "time_budget": 300,
        "config_list": [
            {
                "model": "xgboost",
                "scale_pos_weight": 587, # Ajuste o peso conforme o grau de desbalanceamento
            },
            {
                "model": "lgbm",
                "class_weight": "balanced",
            }
        ],
        "log_file_name": "flaml_cost_sensitive.log",
        "seed": SEED
    }

    print("Iniciando treinamento com FLAML...")
    mem_antes_treino = get_memory_usage_mb()
    inicio_treino = time.perf_counter()

    automl.fit(X_train=X_train, y_train=y_train, **settings)

    fim_treino = time.perf_counter()
    tempo_treino_total = round(fim_treino - inicio_treino, 2)
    mem_depois_treino = get_memory_usage_mb()

    # =========================
    # RESULTADOS DO MELHOR MODELO
    # =========================
    print("\n===== MELHOR RESULTADO FLAML =====")
    print("Melhor modelo:", automl.best_estimator)
    print("Melhores parâmetros:", automl.best_config)
    print("Melhor loss:", automl.best_loss)

    # =========================
    # SELECIONAR OS 5 MELHORES ESTIMADORES
    # =========================
    perdas_por_estimador = automl.best_loss_per_estimator
    melhores_estimadores = sorted(
        perdas_por_estimador.items(),
        key=lambda item: item[1]
    )[:5]

    if not melhores_estimadores:
        raise RuntimeError("O FLAML não encontrou estimadores avaliados.")

    print("\n===== 5 MELHORES ESTIMADORES (VALIDAÇÃO FLAML) =====")
    print(pd.DataFrame(
        melhores_estimadores,
        columns=["Modelo", "Loss validação"]
    ).to_string(index=False))

    # =========================
    # MÉTRICAS DOS 5 MELHORES NO TESTE
    # =========================
    resultados = []
    y_true = y_test.to_numpy()

    for rank, (nome_modelo, loss_validacao) in enumerate(melhores_estimadores, start=1):
        modelo = automl.best_model_for_estimator(nome_modelo)
        
        # Tempo de inferência para cada modelo
        inicio_inferencia = time.perf_counter()
        y_pred = modelo.predict(X_test)

        if hasattr(modelo, "predict_proba"):
            probabilidades = modelo.predict_proba(X_test)
            classes = list(modelo.classes_)
            y_score = probabilidades[:, classes.index(1)]
        elif hasattr(modelo, "decision_function"):
            y_score = modelo.decision_function(X_test)
            if list(modelo.classes_).index(1) == 0:
                y_score = -y_score
        else:
            y_score = y_pred
            
        tempo_inferencia = time.perf_counter() - inicio_inferencia

        resultados.append({
            "Posição": rank,
            "Modelo": nome_modelo,
            "Loss validação": round(float(loss_validacao), 4),
            "Accuracy": round(accuracy_score(y_true, y_pred), 4),
            "Precision": round(precision_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
            "Recall": round(recall_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
            "F1-score Weighted": round(f1_score(y_true, y_pred, average="weighted", zero_division=0), 4),
            "F1-score": round(f1_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
            "Average Precision": round(average_precision_score(y_true, y_score), 4),
            "Seed": SEED,
            "Tempo Inferência (s)": round(tempo_inferencia, 4),
            "Tempo Treino Total (s)": tempo_treino_total,
            "Uso RAM Treino (MB)": mem_depois_treino
        })

    df_resultados_flaml = pd.DataFrame(resultados)
    print("\n===== MÉTRICAS DOS 5 MELHORES MODELOS FLAML (TESTE) =====")
    print(df_resultados_flaml.to_string(index=False))